# Task 2 spatial failure playground

Task 2 can fail in at least two very different ways: the global point cloud can be wrong, or the right biological states can be placed in the wrong locations. This notebook keeps the public target fixed and perturbs one of those pieces at a time.

The transformations are defined once in [`experiments.py`](experiments.py) and reused by the result generator.

In [ ]:
!pip -q install "git+https://github.com/aristoteleo/veckit.git@46d41e63f42a9aab815db20b742feeccd249cb17" pandas matplotlib

In [ ]:
from pathlib import Path
import sys, urllib.request
import anndata as ad
import numpy as np

LAB = Path.cwd() / 'intuition-lab'
if not (LAB / 'experiments.py').exists(): LAB = Path.cwd()
sys.path.insert(0, str(LAB))
from experiments import make_t2_failures, score_t2_failures

DATA = Path('/content/vec_t2_failures')
DATA.mkdir(exist_ok=True)
base = 'https://raw.githubusercontent.com/aristoteleo/veckit/46d41e63f42a9aab815db20b742feeccd249cb17/data/'
for name in ['sample_heart_9.25.h5ad', 'sample_heart_9.5.h5ad']:
    p = DATA / name
    if not p.exists(): urllib.request.urlretrieve(base + name, p)

ref_path = DATA / 'sample_heart_9.25.h5ad'
target_path = DATA / 'sample_heart_9.5.h5ad'
target = ad.read_h5ad(target_path)
print('target:', target.shape, '| coords:', target.obsm['spatial_3D'].shape)

## Controlled failures

The geometry controls are translation, a proper 67° rotation, reflection, 2× uniform scale, and anisotropic stretch. The last control keeps the **point cloud exactly unchanged** and only shuffles expression states among those coordinates.

That last one is the cleanest way to separate tissue shape from local biological organization.

In [ ]:
preds, meta = make_t2_failures(target, DATA / 'predictions', seed=0)
print('built:', list(preds))
print('proper rotation:', meta['proper_rotation_degrees'], 'degrees')

In [ ]:
scores = score_t2_failures(preds, target=target_path, reference=ref_path)
scores[['d2_shape', 'sliced_wasserstein', 'occupancy_dice', 'scale_log_ratio', 'neighborhood_mmd', 'morans_I_agreement']].round(5)

## The proper-rotation row is a scorer audit, not a model failure

A proper rigid rotation should not alter a rigid-frame-invariant shape score. In the pinned public scorer, however, some proper rotations get a nonzero sliced-Wasserstein penalty and reduced occupancy Dice.

A separate sweep over 72 z-axis rotations plus 50 random SO(3) rotations found that the failures line up **exactly** with opposite-handed PCA canonical frames. The PCA axes are stable; the problem is the arbitrary SVD sign choice combined with downstream search over determinant-+1 flips only. See [`rotation_audit.py`](rotation_audit.py), the committed CSV/summary, and [veckit issue #7](https://github.com/aristoteleo/veckit/issues/7).

This is separate from the documented fact that `d2_shape` is reflection-blind.

## What I would remember

A good spatial model needs more than the right silhouette. It can have the right global shape while putting the wrong cell states next to each other. And when a supposedly harmless coordinate-frame change moves a score, audit the scorer before concluding the model got worse.